In [1]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score

SEED = 42               # makes every random step repeatable
BASELINE_CHURN = 0.05   # static churn assumption used by the CLV portal today
HORIZON_YEARS = 15      # CLV horizon from the brief
MODEL_VERSION = "v1.0"
RUN_ID = "RUN_01"

/Users/rameshmandava/Desktop/NewPURE/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Load the raw data
df = pd.read_csv("data/raw/Case_Study_2_CLV_Predictive_Analytics.csv", parse_dates=["contract_end_date"])

# Snapshot date = earliest contract end date in the data (used to turn dates into day counts)
SCORE_DATE = df["contract_end_date"].min()

print(df.shape)
print("Score date:", SCORE_DATE.date())
print(df.dtypes)
df.head()

(2000, 9)
Score date: 2026-10-01
account_id                                object
array_id                                  object
contract_end_date                 datetime64[ns]
historical_15yr_net_profit_usd             int64
current_capacity_tb                        int64
utilization_rate_pct                     float64
static_baseline_churn_rate               float64
churn_event_next_12m                     float64
expansion_potential_signal                object
dtype: object


,account_id,array_id,contract_end_date,historical_15yr_net_profit_usd,current_capacity_tb,utilization_rate_pct,static_baseline_churn_rate,churn_event_next_12m,expansion_potential_signal
0,ACC_1000,ARR_5000,2026-10-01,146797,1382,0.59,0.05,0.0,Medium
1,ACC_1001,ARR_5001,2026-10-01,1236408,1809,0.93,0.05,1.0,UNKNOWN
2,ACC_1002,ARR_5002,2026-10-02,2790632,605,1.28,0.05,0.0,Low
3,ACC_1003,ARR_5003,2026-10-02,1233288,1567,0.47,0.05,0.0,Low
4,ACC_1004,ARR_5004,2026-10-03,2881167,488,1.22,0.05,0.0,Low


In [3]:
# Data quality audit: find the 4 injected anomalies before touching anything
print("Rows:", len(df), "| accounts:", df["account_id"].nunique(), "| arrays:", df["array_id"].nunique())
print("Duplicate rows:", df.duplicated().sum())
print("Static baseline values:", df["static_baseline_churn_rate"].unique())
print()
print("1) Utilization above 100%:", (df["utilization_rate_pct"] > 1).sum(), "rows, max =", df["utilization_rate_pct"].max())
print("2) Negative net profit:", (df["historical_15yr_net_profit_usd"] < 0).sum(), "rows, min =", df["historical_15yr_net_profit_usd"].min())
print("3) Missing churn label:", df["churn_event_next_12m"].isna().sum(), "rows")
print("4) Expansion signal values:")
print(df["expansion_potential_signal"].value_counts())

Rows: 2000 | accounts: 1000 | arrays: 2000
Duplicate rows: 0
Static baseline values: [0.05]

1) Utilization above 100%: 862 rows, max = 1.45
2) Negative net profit: 58 rows, min = -93780
3) Missing churn label: 116 rows
4) Expansion signal values:
expansion_potential_signal
Low        841
Medium     622
High       426
UNKNOWN    111
Name: count, dtype: int64


In [4]:
# Feature pipeline. No row is dropped: each anomaly is kept and turned into features.
def build_features(data):
    features = pd.DataFrame(index=data.index)

    # Utilization is a fraction of capacity, so values above 1.0 (100%) are out of bounds.
    # 43% of rows are above 1.0, so capping alone would flatten them all to the same value.
    # Keep the capped value, a flag, and how far above 1.0 the array is.
    util = data["utilization_rate_pct"]
    features["utilization_capped"] = util.clip(upper=1.0)
    features["util_over_100_flag"] = (util > 1.0).astype(int)
    features["util_excess"] = (util - 1.0).clip(lower=0)

    # log(x) fails on negative profit, so use a signed log: sign(x) * log(1 + |x|).
    profit = data["historical_15yr_net_profit_usd"]
    features["profit_signed_log"] = np.sign(profit) * np.log1p(profit.abs())
    features["is_loss_making"] = (profit < 0).astype(int)

    features["capacity_tb"] = data["current_capacity_tb"]
    features["days_to_contract_end"] = (data["contract_end_date"] - SCORE_DATE).dt.days

    # UNKNOWN is kept as its own category instead of being filled with the most common value.
    for level in ["Low", "Medium", "High", "UNKNOWN"]:
        features["expansion_" + level] = (data["expansion_potential_signal"] == level).astype(int)
    return features


# Cohorts for the back-test. 
df["cohort_contract_end_year"] = df["contract_end_date"].dt.year
df["cohort_capacity_tier"] = pd.qcut(df["current_capacity_tb"], 3, labels=["Entry", "Mid", "Enterprise"])
df["cohort_expansion_signal"] = df["expansion_potential_signal"]
df["cohort_utilization"] = np.where(df["utilization_rate_pct"] > 1.0, "Over 100%", "Up to 100%")
df["cohort_profit"] = np.where(df["historical_15yr_net_profit_usd"] < 0, "Loss-making", "Profitable")

X_all = build_features(df)
X_all.describe().T

,count,mean,std,min,25%,50%,75%,max
utilization_capped,2000.0,0.821545,0.210696,0.350000,0.6600,0.930000,1.000000,1.000000
util_over_100_flag,2000.0,0.431000,0.495340,0.000000,0.0000,0.000000,1.000000,1.000000
util_excess,2000.0,0.095495,0.138026,0.000000,0.0000,0.000000,0.180000,0.450000
profit_signed_log,2000.0,13.584928,4.270333,-11.448718,13.8122,14.573663,15.002512,15.319327
is_loss_making,2000.0,0.029000,0.167848,0.000000,0.0000,0.000000,0.000000,1.000000
capacity_tb,2000.0,1282.771000,699.754635,53.000000,673.5000,1305.500000,1880.000000,2498.000000
days_to_contract_end,2000.0,499.500000,288.747186,0.000000,249.7500,499.500000,749.250000,999.000000
expansion_Low,2000.0,0.420500,0.493763,0.000000,0.0000,0.000000,1.000000,1.000000
expansion_Medium,2000.0,0.311000,0.463019,0.000000,0.0000,0.000000,1.000000,1.000000
expansion_High,2000.0,0.213000,0.409530,0.000000,0.0000,0.000000,0.000000,1.000000


In [5]:
# Isolate the 116 rows with no label BEFORE any training. They become the scoring set.
labelled = df[df["churn_event_next_12m"].notna()].copy()
unlabelled = df[df["churn_event_next_12m"].isna()].copy()

X = X_all.loc[labelled.index]
y = labelled["churn_event_next_12m"].astype(int)
groups = labelled["account_id"]   # each account has 2 arrays; they must stay in the same fold

print("Labelled:", len(labelled), "| unlabelled (scoring set):", len(unlabelled))
print("Actual churn rate:", round(y.mean(), 4))

Labelled: 1884 | unlabelled (scoring set): 116
Actual churn rate: 0.1582
